# EduPro -- Student Segmentation & Personalized Course Recommendation System

**Author:** Final Year B.E. CSE Project
**Notebook purpose:** Interactive, cell-by-cell walkthrough of the ML pipeline implemented
in `train_model.py` and `recommendation.py`, for exploration, viva demonstration, and
research-paper evidence.

## Contents
1. Load & Inspect Raw Data
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Preprocessing (Encoding + Scaling)
5. Elbow Method for Optimal K
6. K-Means Clustering
7. Cluster Profiling & Naming
8. Recommendation Engine Demo
9. Conclusions & Insights


## 1. Load & Inspect Raw Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

users = pd.read_csv("../data/users.csv")
courses = pd.read_csv("../data/courses.csv")
transactions = pd.read_csv("../data/transactions.csv")
transactions["TransactionDate"] = pd.to_datetime(transactions["TransactionDate"])

print("Users:", users.shape)
print("Courses:", courses.shape)
print("Transactions:", transactions.shape)
users.head()

In [ ]:
courses.head()

In [ ]:
transactions.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print(users.isnull().sum())
print("---")
print(transactions.isnull().sum())

In [ ]:
users["Age"].plot(kind="hist", bins=20, title="Age Distribution", color="#4F46E5")
plt.xlabel("Age")
plt.show()

In [ ]:
users["Gender"].value_counts().plot(kind="pie", autopct="%1.1f%%", title="Gender Distribution")
plt.ylabel("")
plt.show()

In [ ]:
merged = transactions.merge(courses, on="CourseID", how="left")
merged["CourseCategory"].value_counts().plot(kind="barh", title="Enrollments by Category", color="#10B981")
plt.show()

In [ ]:
print(f"Total Revenue: Rs. {transactions['Amount'].sum():,.2f}")
print(f"Average Transaction Value: Rs. {transactions['Amount'].mean():,.2f}")
print(f"Unique active learners: {transactions['UserID'].nunique()} / {len(users)} total users")

## 3. Feature Engineering

We aggregate transaction-level data into ONE row per learner with Engagement, Preference, and Behavioural features -- see `train_model.py::build_learner_features` for the full, reusable implementation.

In [ ]:
import sys
sys.path.append("..")
from train_model import build_learner_features, encode_categorical, scale_features, FEATURE_COLUMNS

learner_df = build_learner_features(users, courses, transactions)
print(learner_df.shape)
learner_df.head()

## 4. Preprocessing: Encoding + Scaling

In [ ]:
learner_df, encoders = encode_categorical(learner_df)
print("Encoders:", encoders)

X_scaled, scaler = scale_features(learner_df)
print("Scaled feature matrix shape:", X_scaled.shape)

## 5. Elbow Method for Optimal K

**Why the Elbow Method?** K-Means requires us to choose K upfront. Inertia (within-cluster sum of squares) always decreases as K grows, so instead we look for the point of diminishing returns -- the 'elbow' -- and cross-validate with the Silhouette Score, which measures how well-separated the resulting clusters are.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertias, silhouettes = [], []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, ax1 = plt.subplots(figsize=(8,5))
ax1.plot(list(K_range), inertias, marker="o", color="#4F46E5")
ax1.set_xlabel("K"); ax1.set_ylabel("Inertia", color="#4F46E5")
ax2 = ax1.twinx()
ax2.plot(list(K_range), silhouettes, marker="s", color="#10B981")
ax2.set_ylabel("Silhouette Score", color="#10B981")
plt.title("Elbow Method & Silhouette Score")
plt.show()

## 6. K-Means Clustering (Final Model)

In [ ]:
best_k = 3  # chosen based on the elbow/silhouette analysis above
model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
learner_df["Cluster"] = model.fit_predict(X_scaled)
print("Silhouette score:", silhouette_score(X_scaled, learner_df["Cluster"]))
learner_df["Cluster"].value_counts()

## 7. Cluster Profiling & Naming

In [ ]:
from train_model import assign_cluster_names
learner_df["ClusterName"] = assign_cluster_names(learner_df)
profile = learner_df.groupby("ClusterName")[FEATURE_COLUMNS[:9]].mean().round(2)
profile

In [ ]:
from sklearn.decomposition import PCA
coords = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
plt.figure(figsize=(8,6))
for name in learner_df["ClusterName"].unique():
    mask = learner_df["ClusterName"] == name
    plt.scatter(coords[mask,0], coords[mask,1], label=name, alpha=0.7, s=20)
plt.legend(); plt.title("PCA Projection of Learner Segments")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

## 8. Recommendation Engine Demo

In [ ]:
from recommendation import load_data, recommend_for_user

u, c, t, clustered = load_data("../data")
sample_user = clustered["UserID"].iloc[0]
recs = recommend_for_user(sample_user, u, c, t, clustered, top_n=5)
recs

## 9. Conclusions & Insights

- Three statistically distinct, business-interpretable learner segments emerged:
  **Explorers**, **Deep Specialists**, and **Premium / Career-Focused Learners**.
- Spending behaviour and category diversity were the strongest differentiators
  between segments (see correlation heatmap in `diagrams/eda_correlation_heatmap.png`).
- The cluster-aware hybrid recommender (collaborative + content-based) generates
  relevant, non-redundant course suggestions for every learner segment.
- This notebook mirrors `train_model.py` exactly, so results are 100% reproducible
  end-to-end via `python train_model.py`.
